# RAS 2026 PSC — GNN+RL Blocking Plan Optimizer

**Pipeline:** Greedy → Behavioral Cloning (supervised warm-start) → REINFORCE (policy gradient RL)

**Paper angle:** *Learning Railroad Blocking Plans via Behavioral Cloning and Policy Gradient RL on INFORMS RAS 2026 benchmark*

## 0. Setup

In [ ]:
# Clone repo
import os
if not os.path.exists('ras2026'):
    !git clone https://github.com/nepersoned/ras2026.git
%cd ras2026

In [ ]:
!pip install -q kaggle scipy numpy pandas torch

In [ ]:
# Kaggle API credentials — paste your kaggle.json content here
import json, os
os.makedirs('/root/.kaggle', exist_ok=True)

# TODO: replace with your credentials
kaggle_creds = {"username": "YOUR_USERNAME", "key": "YOUR_API_KEY"}
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

In [ ]:
# Download competition data
!kaggle competitions download -c informs-ras-2026-problem-solving-competition -p . --force
!unzip -q informs-ras-2026-problem-solving-competition.zip -d ras_release_v1.1
!ls ras_release_v1.1/

## 1. Environment

In [ ]:
import sys
sys.path.insert(0, '.')

import math, random, time, json, csv
from pathlib import Path
from collections import defaultdict
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

from solver import (
    load_layer, load_od_matrix, Network, Demand, Solution,
    greedy_construct, evaluate, build_json,
    COMMODITY_TO_BLOCK_TYPE, DIRECT_ONLY,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
BASE = Path('ras_release_v1.1/ras_release_v1.1')

In [ ]:
# ── RAS MDP Environment ────────────────────────────────────────────────────────

BLOCK_TYPES = ['Merchandise', 'Coal', 'Grain', 'Intermodal', 'Automobile']
MAX_HUBS    = 50
FEAT_DIM    = 7 + MAX_HUBS * 4   # 207
N_ACTIONS   = MAX_HUBS + 2       # 0=direct, 1..MAX_HUBS=hub, MAX_HUBS+1=unserved


def load_env(layer: str, multiplier: float):
    """Load all data and precompute network for one scenario."""
    nodes_df, links_df, demands_scaled, demands_raw, settings = \
        load_layer(layer, multiplier)

    yard_rows = nodes_df[nodes_df['node_type'] == 'yard']
    yard_info = {
        int(r['node_id']): {
            'num_tracks':        float(r.get('num_tracks',        9999) or 9999),
            'handling_capacity': float(r.get('handling_capacity', 1e9)  or 1e9),
            'handling_cost':     float(r.get('handling_cost',     0)    or 0),
        }
        for _, r in yard_rows.iterrows()
    }

    origin_ids   = set(demands_scaled['origin_yard_id'].astype(int))
    dest_ids     = set(demands_scaled['dest_yard_id'].astype(int))
    all_yard_ids = sorted(origin_ids | dest_ids)

    all_od_pairs = {(o, d) for o in all_yard_ids for d in all_yard_ids if o != d}
    od_matrix    = load_od_matrix(all_od_pairs)

    net = Network(nodes_df, links_df, origin_ids, dest_ids, settings, verbose=True)

    demands = [
        Demand(
            idx            = idx,
            demand_id      = int(row['demand_id']),
            commodity_type = str(row['block_type']),
            block_type     = COMMODITY_TO_BLOCK_TYPE.get(str(row['block_type']), 'Manifest'),
            origin         = int(row['origin_yard_id']),
            dest           = int(row['dest_yard_id']),
            volume         = int(row['volume']),
            sp_dist        = od_matrix.get(
                (int(row['origin_yard_id']), int(row['dest_yard_id'])),
                net.dist(int(row['origin_yard_id']), int(row['dest_yard_id']))
            ),
        )
        for idx, (_, row) in enumerate(demands_scaled.iterrows())
    ]

    return dict(
        net=net, od_matrix=od_matrix, settings=settings,
        yard_info=yard_info, demands=demands,
        all_yards=all_yard_ids,
        nodes_df=nodes_df, links_df=links_df, demands_raw=demands_raw,
    )

## 2. Feature Engineering

In [ ]:
def ranked_hubs(dem, env):
    if dem.commodity_type in DIRECT_ONLY:
        return []
    scored = []
    for hub in env['all_yards']:
        if hub == dem.origin or hub == dem.dest:
            continue
        d1 = env['net'].dist(dem.origin, hub)
        d2 = env['net'].dist(hub, dem.dest)
        if math.isinf(d1) or math.isinf(d2):
            continue
        scored.append((d1 + d2, hub))
    scored.sort()
    return [h for _, h in scored]


def demand_feature(dem, hubs, env):
    """207-dim feature vector for one demand."""
    od_d  = env['net'].dist(dem.origin, dem.dest)
    od_ic = env['net'].interchanges(dem.origin, dem.dest)
    ct_oh = [1.0 if dem.commodity_type == t else 0.0 for t in BLOCK_TYPES]
    base  = [math.log1p(dem.volume), math.log1p(od_d), od_ic / 5.0] + ct_oh  # 8 dims... wait
    # actually: log_vol, log_od_dist, od_ic_norm, 4×block_type_onehot → 7 total
    bt_oh = [1.0 if COMMODITY_TO_BLOCK_TYPE.get(dem.commodity_type,'Manifest') == t
             else 0.0
             for t in ['Manifest','Bulk','Intermodal','Multilevel']]
    base  = [math.log1p(dem.volume), math.log1p(od_d if math.isfinite(od_d) else 1e6),
             od_ic / 5.0] + bt_oh   # 7 dims

    hub_feats = [0.0] * (MAX_HUBS * 4)
    for j, hub in enumerate(hubs[:MAX_HUBS]):
        d1  = env['net'].dist(dem.origin, hub)
        d2  = env['net'].dist(hub, dem.dest)
        if math.isinf(d1) or math.isinf(d2):
            continue
        ic1 = env['net'].interchanges(dem.origin, hub)
        ic2 = env['net'].interchanges(hub, dem.dest)
        hc  = env['yard_info'].get(hub, {}).get('handling_cost', 0.0)
        hub_feats[j*4]   = math.log1p(d1)
        hub_feats[j*4+1] = math.log1p(d2)
        hub_feats[j*4+2] = (ic1 + ic2) / 5.0
        hub_feats[j*4+3] = hc / 500.0

    return base + hub_feats   # 207 dims


def action_mask(dem, hubs, env):
    """Boolean mask of valid actions."""
    mask = [False] * N_ACTIONS
    if not math.isinf(env['net'].dist(dem.origin, dem.dest)):
        mask[0] = True
    if dem.commodity_type not in DIRECT_ONLY:
        for j, hub in enumerate(hubs[:MAX_HUBS]):
            d1 = env['net'].dist(dem.origin, hub)
            d2 = env['net'].dist(hub, dem.dest)
            if not math.isinf(d1) and not math.isinf(d2):
                mask[j + 1] = True
    mask[N_ACTIONS - 1] = True  # always can be unserved
    return mask


def action_to_route(action, dem, hubs):
    if action == 0:
        return [(dem.origin, dem.dest)]
    if action == N_ACTIONS - 1:
        return None
    hi = action - 1
    if hi >= len(hubs):
        return None
    return [(dem.origin, hubs[hi]), (hubs[hi], dem.dest)]


def precompute(env):
    """Precompute per-demand features, masks, and hub lists."""
    data = []
    for dem in env['demands']:
        if dem.volume <= 0:
            data.append(None)
            continue
        hubs = ranked_hubs(dem, env)
        feat = demand_feature(dem, hubs, env)
        mask = action_mask(dem, hubs, env)
        data.append({'feat': feat, 'mask': mask, 'hubs': hubs})
    return data


print('Feature engineering ready.')

## 3. Policy Network

In [ ]:
class BlockingPolicy(nn.Module):
    """MLP policy: demand features → action logits.

    For paper: this is a shared-parameter policy over all demands.
    Input: 207-dim demand feature vector
    Output: N_ACTIONS=52 logits (direct / hub_1..hub_50 / unserved)
    """
    def __init__(self, feat_dim=FEAT_DIM, n_actions=N_ACTIONS, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        # x: (B, FEAT_DIM)  →  (B, N_ACTIONS)
        return self.net(x)


def make_policy():
    return BlockingPolicy().to(DEVICE)


print(f'Policy: {FEAT_DIM} → 256 → 256 → {N_ACTIONS}  ({sum(p.numel() for p in make_policy().parameters()):,} params)')

## 4. Behavioral Cloning (warm-start)

In [ ]:
def greedy_action_label(route, hubs):
    """Convert greedy routing to action index."""
    if route is None:
        return N_ACTIONS - 1
    if len(route) == 1:   # direct
        return 0
    hub = route[0][1]
    top = hubs[:MAX_HUBS]
    if hub in top:
        return top.index(hub) + 1
    return N_ACTIONS - 1


def pretrain_bc(policy, env, dd, n_epochs=300, lr=1e-3, print_every=50):
    """Behavioral Cloning: fit policy to greedy solution."""
    # Get greedy solution
    init_sol = greedy_construct(env['demands'], env['net'], env['od_matrix'],
                                env['settings'], env['yard_info'])
    score0, _ = evaluate(init_sol, env['net'], env['od_matrix'],
                         env['settings'], env['yard_info'])

    # Build dataset
    feats, masks, labels = [], [], []
    for i, dem in enumerate(env['demands']):
        if dd[i] is None:
            continue
        route  = init_sol.routings[i]
        label  = greedy_action_label(route, dd[i]['hubs'])
        feats.append(dd[i]['feat'])
        masks.append(dd[i]['mask'])
        labels.append(label)

    feat_t  = torch.tensor(feats,  dtype=torch.float32, device=DEVICE)   # (N, FEAT_DIM)
    label_t = torch.tensor(labels, dtype=torch.long,    device=DEVICE)   # (N,)
    mask_t  = torch.tensor(masks,  dtype=torch.bool,    device=DEVICE)   # (N, N_ACTIONS)

    opt = torch.optim.Adam(policy.parameters(), lr=lr)

    print(f'\n── Behavioral Cloning ──  N={len(labels)}  greedy_score={score0:,.0f}')

    for ep in range(1, n_epochs + 1):
        policy.train()
        logits = policy(feat_t)                              # (N, N_ACTIONS)
        logits = logits.masked_fill(~mask_t, -1e9)
        loss   = F.cross_entropy(logits, label_t)

        opt.zero_grad()
        loss.backward()
        opt.step()

        if ep % print_every == 0:
            with torch.no_grad():
                preds = logits.argmax(dim=1)
                acc   = (preds == label_t).float().mean().item() * 100
            print(f'  ep {ep:4d} | loss={loss.item():.4f} | acc={acc:.1f}%')

    print(f'BC done.')
    return init_sol, score0

## 5. REINFORCE (Policy Gradient RL)

In [ ]:
@torch.no_grad()
def sample_solution(policy, env, dd, greedy=False):
    """Sample (or greedily decode) a complete blocking plan from the policy."""
    policy.eval()
    valid_idxs = [i for i, d in enumerate(dd) if d is not None]
    if not valid_idxs:
        return [None] * len(dd), []

    feats  = torch.tensor([dd[i]['feat'] for i in valid_idxs],
                          dtype=torch.float32, device=DEVICE)
    masks  = torch.tensor([dd[i]['mask'] for i in valid_idxs],
                          dtype=torch.bool,    device=DEVICE)
    logits = policy(feats)                            # (N_valid, N_ACTIONS)
    logits = logits.masked_fill(~masks, -1e9)

    if greedy:
        actions_valid = logits.argmax(dim=1).tolist()
    else:
        actions_valid = Categorical(logits=logits).sample().tolist()

    routings = [None] * len(dd)
    actions  = [N_ACTIONS - 1] * len(dd)
    for j, i in enumerate(valid_idxs):
        a = actions_valid[j]
        actions[i]  = a
        routings[i] = action_to_route(a, env['demands'][i], dd[i]['hubs'])

    return routings, actions


def compute_log_probs(policy, env, dd, actions):
    """Compute sum of log-probs for gradient (differentiable)."""
    policy.train()
    valid_idxs = [i for i, d in enumerate(dd) if d is not None]

    feats  = torch.tensor([dd[i]['feat'] for i in valid_idxs],
                          dtype=torch.float32, device=DEVICE)
    masks  = torch.tensor([dd[i]['mask'] for i in valid_idxs],
                          dtype=torch.bool,    device=DEVICE)
    act_t  = torch.tensor([actions[i] for i in valid_idxs],
                          dtype=torch.long,    device=DEVICE)

    logits = policy(feats)
    logits = logits.masked_fill(~masks, -1e9)
    log_p  = F.log_softmax(logits, dim=1)            # (N_valid, N_ACTIONS)
    return log_p.gather(1, act_t.unsqueeze(1)).squeeze(1).sum()


def train_reinforce(policy, env, dd, n_episodes=1000, lr=1e-4,
                    baseline_decay=0.99, print_every=50):
    """REINFORCE with exponential moving average baseline."""
    opt = torch.optim.Adam(policy.parameters(), lr=lr)

    # Init baseline from greedy
    routings, _ = sample_solution(policy, env, dd, greedy=True)
    sol          = Solution(env['demands'], routings)
    baseline, _  = evaluate(sol, env['net'], env['od_matrix'],
                            env['settings'], env['yard_info'])
    best_score   = baseline
    best_state   = deepcopy(policy.state_dict())

    log = []   # for paper: training curve
    print(f'\n── REINFORCE ──  init={baseline:,.0f}  episodes={n_episodes}')

    for ep in range(1, n_episodes + 1):
        # Sample rollout
        routings, actions = sample_solution(policy, env, dd, greedy=False)
        sol   = Solution(env['demands'], routings)
        score, _ = evaluate(sol, env['net'], env['od_matrix'],
                            env['settings'], env['yard_info'])

        advantage = baseline - score   # positive → good move
        baseline  = baseline_decay * baseline + (1 - baseline_decay) * score

        # Policy gradient
        log_p = compute_log_probs(policy, env, dd, actions)
        loss  = -advantage * log_p   # maximise advantage × log_prob

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        opt.step()

        if score < best_score:
            best_score = score
            best_state = deepcopy(policy.state_dict())

        log.append({'ep': ep, 'sample': score, 'baseline': baseline, 'best': best_score})

        if ep % print_every == 0:
            routings_g, _ = sample_solution(policy, env, dd, greedy=True)
            sol_g = Solution(env['demands'], routings_g)
            g, _  = evaluate(sol_g, env['net'], env['od_matrix'],
                             env['settings'], env['yard_info'])
            print(f'  ep {ep:4d} | sample={score:>14,.0f} | greedy={g:>14,.0f} | best={best_score:>14,.0f}')

    policy.load_state_dict(best_state)
    print(f'  Best score: {best_score:,.0f}')
    return best_score, log

## 6. Train on L1 → Transfer to L2/L3

In [ ]:
CASE_ORDER = [
    ('l1', 0.5), ('l1', 1.0), ('l1', 2.0),
    ('l2', 0.5), ('l2', 1.0), ('l2', 2.0),
    ('l3', 0.5), ('l3', 1.0), ('l3', 2.0),
]

# Training hyperparams — tune for paper ablation
BC_EPOCHS      = 300
RL_EPISODES    = {'l1': 2000, 'l2': 1000, 'l3': 500}
TRANSFER_LR    = 1e-4   # fine-tune LR for L2/L3
TRANSFER_EPS   = {'l2': 500, 'l3': 200}

solutions   = {}   # layer_mult → json dict
train_logs  = {}   # layer_mult → log list
models      = {}   # layer → policy

In [ ]:
# ── L1 Training (train fresh policy for each multiplier) ──────────────────────

import os
os.makedirs('saved_models', exist_ok=True)

for mult in [0.5, 1.0, 2.0]:
    tag = f'l1_{mult}'
    print(f'\n{"="*60}')
    print(f'  L1 × {mult}')
    print(f'{"="*60}')

    t0  = time.time()
    env = load_env('l1', mult)
    dd  = precompute(env)

    policy = make_policy()

    # Phase 1: BC
    init_sol, g0 = pretrain_bc(policy, env, dd, n_epochs=BC_EPOCHS)

    # Phase 2: REINFORCE
    best, log = train_reinforce(policy, env, dd, n_episodes=RL_EPISODES['l1'])
    train_logs[tag] = log

    # Greedy decode final policy
    routings, _ = sample_solution(policy, env, dd, greedy=True)
    sol          = Solution(env['demands'], routings)
    score, stats = evaluate(sol, env['net'], env['od_matrix'],
                            env['settings'], env['yard_info'])
    print(f'  Final: stress={stats["stress_score"]:,.0f}  blocks={stats["n_blocks"]}')

    solutions[tag] = build_json(sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    torch.save(policy.state_dict(), f'saved_models/policy_l1_{mult}.pt')
    models[f'l1_{mult}'] = policy

    print(f'  Elapsed: {time.time()-t0:.1f}s')

In [ ]:
# ── L2 Training (fine-tune from L1 1.0× model) ────────────────────────────────

for mult in [0.5, 1.0, 2.0]:
    tag = f'l2_{mult}'
    print(f'\n{"="*60}')
    print(f'  L2 × {mult}')
    print(f'{"="*60}')

    t0  = time.time()
    env = load_env('l2', mult)
    dd  = precompute(env)

    # Transfer: start from L1 1.0× weights
    policy = make_policy()
    policy.load_state_dict(torch.load('saved_models/policy_l1_1.0.pt', map_location=DEVICE))
    print('  Loaded L1 weights (transfer learning)')

    # BC fine-tune
    init_sol, g0 = pretrain_bc(policy, env, dd, n_epochs=100, print_every=25)

    # REINFORCE fine-tune
    best, log = train_reinforce(policy, env, dd,
                                n_episodes=TRANSFER_EPS['l2'], lr=TRANSFER_LR)
    train_logs[tag] = log

    routings, _ = sample_solution(policy, env, dd, greedy=True)
    sol          = Solution(env['demands'], routings)
    score, stats = evaluate(sol, env['net'], env['od_matrix'],
                            env['settings'], env['yard_info'])
    print(f'  Final: stress={stats["stress_score"]:,.0f}  blocks={stats["n_blocks"]}')

    solutions[tag] = build_json(sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    torch.save(policy.state_dict(), f'saved_models/policy_l2_{mult}.pt')

    print(f'  Elapsed: {time.time()-t0:.1f}s')

In [ ]:
# ── L3 Training (fine-tune from L2 1.0× model) ────────────────────────────────

for mult in [0.5, 1.0, 2.0]:
    tag = f'l3_{mult}'
    print(f'\n{"="*60}')
    print(f'  L3 × {mult}')
    print(f'{"="*60}')

    t0  = time.time()
    env = load_env('l3', mult)
    dd  = precompute(env)

    policy = make_policy()
    policy.load_state_dict(torch.load('saved_models/policy_l2_1.0.pt', map_location=DEVICE))
    print('  Loaded L2 weights (transfer learning)')

    # BC fine-tune (light)
    init_sol, g0 = pretrain_bc(policy, env, dd, n_epochs=50, print_every=10)

    # REINFORCE fine-tune
    best, log = train_reinforce(policy, env, dd,
                                n_episodes=TRANSFER_EPS['l3'], lr=TRANSFER_LR)
    train_logs[tag] = log

    routings, _ = sample_solution(policy, env, dd, greedy=True)
    sol          = Solution(env['demands'], routings)
    score, stats = evaluate(sol, env['net'], env['od_matrix'],
                            env['settings'], env['yard_info'])
    print(f'  Final: stress={stats["stress_score"]:,.0f}  blocks={stats["n_blocks"]}')

    solutions[tag] = build_json(sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    torch.save(policy.state_dict(), f'saved_models/policy_l3_{mult}.pt')

    print(f'  Elapsed: {time.time()-t0:.1f}s')

## 7. Generate submission.csv

In [ ]:
rows = [['ID', 'data']]

for case_id, (layer, mult) in enumerate(CASE_ORDER):
    tag = f'{layer}_{mult}'
    sol = solutions.get(tag)
    if sol is None:
        print(f'[ID {case_id}] {tag} — MISSING, using empty')
        rows.append([case_id, '{}'])
        continue
    data_str = json.dumps(sol, separators=(',', ':'))
    n_blocks  = len(sol['outputs']['1 Block Design'])
    n_seqs    = len(sol['outputs']['2 Blocking Sequence'])
    print(f'[ID {case_id}] {tag}  blocks={n_blocks}  seqs={n_seqs}  len={len(data_str):,}')
    rows.append([case_id, data_str])

with open('submission.csv', 'w', newline='', encoding='utf-8') as f:
    csv.writer(f).writerows(rows)

print(f'\nWrote submission.csv  ({sum(len(r[1]) for r in rows[1:])/1e6:.1f} MB)')

In [ ]:
# Download
from google.colab import files
files.download('submission.csv')

## 8. Training Curves (for paper)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (layer, mult) in zip(axes, [('l1', 1.0), ('l2', 1.0), ('l3', 1.0)]):
    tag = f'{layer}_{mult}'
    log = train_logs.get(tag, [])
    if not log:
        continue
    eps   = [d['ep']   for d in log]
    bests = [d['best'] for d in log]
    ax.plot(eps, bests)
    ax.set_title(f'{layer.upper()} ×{mult}')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Stress Score')
    ax.grid(True, alpha=0.3)

plt.suptitle('REINFORCE Training Curves — Best Stress Score per Episode')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('Saved training_curves.png')